# Notebook 12 — Datetime Functions Basics

**Datasets:** `samples.nyctaxi.trips`, `samples.bakehouse.sales_transactions`, `samples.wanderbricks.bookings`  
**Difficulty:** Easy  
**Topics:** `year`, `month`, `dayofweek`, `hour`, `datediff`, `date_add`, `date_trunc`, `to_date`, `date_format`, `current_date`

Each problem teaches a core PySpark datetime function through a practical example.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = SparkSession.builder.getOrCreate()

trips = spark.read.table("samples.nyctaxi.trips")
transactions = spark.read.table("samples.bakehouse.sales_transactions")
bookings = spark.read.table("samples.wanderbricks.bookings")

print("nyctaxi.trips schema:")
trips.printSchema()
print("sales_transactions schema:")
transactions.printSchema()
print("wanderbricks.bookings schema:")
bookings.printSchema()

## Learn — Datetime Functions

| Function | What it does |
|----------|-------------|
| `F.year(col)` / `F.month(col)` / `F.dayofmonth(col)` | Extract date parts |
| `F.hour(col)` / `F.minute(col)` | Extract time parts from a timestamp |
| `F.dayofweek(col)` | Day of week: 1 = Sunday, 7 = Saturday |
| `F.datediff(end_col, start_col)` | Days between two dates (positive = end is later) |
| `F.date_add(col, n)` | Add n days to a date |
| `F.date_trunc("month", col)` | Truncate to start of month (also: "year", "week", "day") |
| `F.date_format(col, "yyyy-MM-dd")` | Format date as string |
| `F.to_date(col)` / `F.to_timestamp(col)` | Parse string to date/timestamp |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

> Use `F.to_timestamp()` for columns that include a time component; use `F.to_date()` for date-only columns.

In [ ]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.nyctaxi.trips")

# Extract time parts from a timestamp
df.select(
    F.col("tpep_pickup_datetime"),
    F.hour("tpep_pickup_datetime").alias("pickup_hour"),
    F.dayofweek("tpep_pickup_datetime").alias("day_of_week")  # 1=Sun, 7=Sat
).show(5)

# Trip duration in minutes (not a problem question — just shows datediff pattern)
df.select(
    F.round(
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60,
        1
    ).alias("duration_mins")
).show(5)

## Problem 1 — Extract Date Components

On `nyctaxi.trips`, extract individual date and time components from `tpep_pickup_datetime` using:
- `F.year()`, `F.month()`, `F.dayofmonth()`, `F.hour()`, `F.minute()`

This is useful for aggregations like "how many trips per hour of day" or "which months are busiest".

**Expected output columns:** `tpep_pickup_datetime`, `year`, `month`, `day`, `hour`, `minute`

In [ ]:
result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'tpep_pickup_datetime' in cols, "Missing column: tpep_pickup_datetime"
assert 'year' in cols, "Missing column: year"
assert 'month' in cols, "Missing column: month"
assert 'day' in cols, "Missing column: day"
assert 'hour' in cols, "Missing column: hour"
assert 'minute' in cols, "Missing column: minute"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — Calculate Trip Duration

On `nyctaxi.trips`, compute `duration_minutes` as the difference between `tpep_dropoff_datetime` and `tpep_pickup_datetime` in minutes.

**Approach:** Cast both timestamps to `long` (Unix epoch seconds), subtract, then divide by 60:
```python
(F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60
```

Filter to rows where `duration_minutes > 0`.

**Expected output columns:** `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `duration_minutes`, `trip_distance`, `fare_amount`

In [ ]:
result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'tpep_pickup_datetime' in cols, "Missing column: tpep_pickup_datetime"
assert 'tpep_dropoff_datetime' in cols, "Missing column: tpep_dropoff_datetime"
assert 'duration_minutes' in cols, "Missing column: duration_minutes"
assert 'trip_distance' in cols, "Missing column: trip_distance"
assert 'fare_amount' in cols, "Missing column: fare_amount"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3 — Day of Week Analysis

On `sales_transactions`, use `F.dayofweek()` on `dateTime` to get a numeric day (1=Sunday … 7=Saturday in Spark). Map these numbers to day names using a `F.when()` chain. Group by `day_name` and compute count and total revenue.


**Expected output columns:** `day_name`, `transaction_count`, `total_revenue`

In [ ]:
result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'day_name' in cols, "Missing column: day_name"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — Stay Duration for Bookings

On `wanderbricks.bookings`, compute the number of nights for each booking using `F.datediff(check_out, check_in)`. Then group by `status` and compute aggregate statistics.


**Expected output columns:** `status`, `avg_nights`, `min_nights`, `max_nights`, `booking_count`

In [ ]:
result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'status' in cols, "Missing column: status"
assert 'avg_nights' in cols, "Missing column: avg_nights"
assert 'min_nights' in cols, "Missing column: min_nights"
assert 'max_nights' in cols, "Missing column: max_nights"
assert 'booking_count' in cols, "Missing column: booking_count"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5 — Date Formatting and Truncation

On `sales_transactions`, use `F.date_trunc("month", col)` to snap `dateTime` to the start of each month. Then use `F.date_format()` to render it as a readable `"yyyy-MM"` string for the `month_label` column. Group by `month_label` to get monthly transaction counts and revenue.


**Expected output columns:** `month_label`, `transaction_count`, `total_revenue`

In [ ]:
result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'month_label' in cols, "Missing column: month_label"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")